## CRIAR DATABASE NO GLUE CATALOG

In [1]:
import boto3

glue = boto3.client("glue", region_name="us-east-1")

glue.create_database(
    DatabaseInput={
        "Name": "siorg",
        "Description": "Tabelas Iceberg do trabalho de BDM"
    }
)

print("Database criado")

Database criado


In [2]:
glue.get_databases()

{'DatabaseList': [{'Name': 'siorg',
   'Description': 'Tabelas Iceberg do trabalho de BDM',
   'CreateTime': datetime.datetime(2026, 7, 13, 15, 14, 55, tzinfo=tzlocal()),
   'CreateTableDefaultPermissions': [{'Principal': {'DataLakePrincipalIdentifier': 'IAM_ALLOWED_PRINCIPALS'},
     'Permissions': ['ALL']}],
   'CatalogId': '931680509227'}],
 'ResponseMetadata': {'RequestId': '8dfa688d-52d9-472f-a498-09f1911e365f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 13 Jul 2026 18:15:08 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '382',
   'connection': 'keep-alive',
   'x-amzn-requestid': '8dfa688d-52d9-472f-a498-09f1911e365f',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

## TESTES TRINO

In [3]:
from trino.dbapi import connect

conn = connect(
    host="localhost",
    port=8090,
    user="cassio",
)

cursor = conn.cursor()

cursor.execute("SHOW CATALOGS")
print(cursor.fetchall())

[['iceberg'], ['system']]


In [4]:
cursor.execute("SHOW SCHEMAS FROM iceberg")
print(cursor.fetchall())

[['information_schema'], ['siorg'], ['system']]


## TESTES S3

In [7]:
cursor.execute("""
DROP SCHEMA IF EXISTS iceberg.siorg
""")

In [8]:
cursor.execute("""
CREATE SCHEMA iceberg.siorg
WITH (
    location = 's3://unb-bdm-siorg/iceberg/siorg/'
)
""")

print("Schema criado com localização no S3")

Schema criado com localização no S3


In [9]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS iceberg.siorg.teste_conexao (
    id BIGINT,
    nome VARCHAR
)
WITH (
    format = 'PARQUET'
)
""")

print("Tabela criada")

Tabela criada


In [10]:
cursor.execute("""
CREATE SCHEMA iceberg.siorg_iceberg
WITH (
    location = 's3://unb-bdm-siorg/iceberg/siorg/'
)
""")

In [11]:
cursor.execute("SHOW TABLES FROM iceberg.siorg")
print(cursor.fetchall())

[['teste_conexao']]


In [12]:
cursor.execute("""
INSERT INTO iceberg.siorg.teste_conexao
VALUES
    (1, 'trino'),
    (2, 'iceberg'),
    (3, 's3')
""")

print(cursor.fetchall())

[[3]]


In [13]:
cursor.execute("""
SELECT *
FROM iceberg.siorg.teste_conexao
ORDER BY id
""")

print(cursor.fetchall())

[[1, 'trino'], [2, 'iceberg'], [3, 's3']]
